# SQLite to Snowflake ETL (Colab-Style Workbook)

## Overview
This notebook moves data from a local SQLite `.db` file into Snowflake in a reproducible, step-by-step flow.

### What you will do
- Discover SQLite tables
- Export each table to CSV
- Create matching Snowflake target tables
- Upload files to an internal Snowflake stage
- Load data with `COPY INTO`


## Quick Start

### Before running
1. Set `SQLITE_DB_PATH` to your local `.db` file.
2. Fill Snowflake credentials (or use env vars).
3. Run cells top-to-bottom.

### Required packages
- `pandas`
- `snowflake-connector-python`


## Local PC Notes

### Finding your project folder
If you're running this on your own computer, use your local repo path (not `/workspace/...`).

Example:
- macOS/Linux: `cd ~/Downloads/soccer_codex`
- Windows PowerShell: `cd C:\Users\<you>\Downloads\soccer_codex`


## 0) Environment Setup

### Install dependencies (optional in Colab/local notebooks)


In [1]:
# Optional: install dependencies if needed
!pip install -q pandas snowflake-connector-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is inco

## 1) Imports

### Import Python libraries


In [2]:
from pathlib import Path
import os
import sqlite3
import pandas as pd
import snowflake.connector
import json
import requests


## 2) Colab DB Upload (optional)

### Upload your `.db` file directly in Google Colab
Run the next cell to upload a SQLite `.db` file from your computer.
If uploaded, `SQLITE_DB_PATH` will be set automatically.


In [3]:
# Colab upload helper (safe to run outside Colab; it will just skip)
UPLOADED_DB_PATH = None
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded.keys()))
        UPLOADED_DB_PATH = Path(first_name).resolve()
        print(f"Uploaded DB detected: {UPLOADED_DB_PATH}")
    else:
        print("No file uploaded. You can still set SQLITE_DB_PATH manually.")
except Exception:
    print("Not running in Colab upload context. Set SQLITE_DB_PATH manually.")


Saving ISE558finaldb.db to ISE558finaldb.db
Uploaded DB detected: /content/ISE558finaldb.db


## 3) Configuration

### Set file paths, table filters, and Snowflake connection settings


In [4]:
# ---------- Local paths ----------
DEFAULT_SQLITE_DB_PATH = Path("/path/to/your/database.db")  # fallback
SQLITE_DB_PATH = UPLOADED_DB_PATH or DEFAULT_SQLITE_DB_PATH
EXPORT_DIR = Path("./sqlite_exports")

# Optional: set to [] to auto-load all tables
TABLES_TO_LOAD = []  # e.g., ["customers", "orders"]

# ---------- Named connection profile ----------
# For SSO login in Colab, use authenticator="externalbrowser".
CONNECTIONS = {
    "my_example_connection": {
        "account": "NWLBALI-XLB64593",
        "user": "AWESOMEIBRAHIM2022",
        #"authenticator": "externalbrowser",
        #"role": "ACCOUNTADMIN",
        "password": "inputpasswordhere",
        "host": "nwlbali-xlb64593.snowflakecomputing.com"
        #"warehouse": "Iu4YZihGC468",
        #"database": "lcy5NJ9fF0Kw",
        #"schema": "clQCSA71GKqU",
    }
}

ACTIVE_CONNECTION = "my_example_connection"
SNOWFLAKE_CONFIG = CONNECTIONS[ACTIVE_CONNECTION].copy()

# Optional bootstrap behavior (create objects if missing)
CREATE_RESOURCES_IF_MISSING = True
WAREHOUSE_SIZE = "XSMALL"
WAREHOUSE_TYPE = "STANDARD"
WAREHOUSE_AUTO_SUSPEND = 60
WAREHOUSE_AUTO_RESUME = True
RESOURCE_NAME_PREFIX = "SQLITE_INGEST"


# ---------- Optional extra source files ----------
INDIRECT_COSTS_XLSX_URL = "https://raw.githubusercontent.com/usc-orchestrated-pipelines-research/ConsultingFirm_DB_OldVersion/main/example_output/spreadsheets/indirect_costs.xlsx"
CLIENT_FEEDBACK_JSON_URL = "https://raw.githubusercontent.com/usc-orchestrated-pipelines-research/ConsultingFirm_DB_OldVersion/main/example_output/json/client_feedbacks.json"
MANUAL_NON_BILLABLE_FILENAME = "non_billable.xlsx"  # upload manually in Colab
EXTRA_SOURCE_DIR = Path("./extra_sources")
EXTRA_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

# Snowflake stage for CSV upload
STAGE_NAME = "SQLITE_INGEST_STAGE"

# Snowflake table naming strategy
TARGET_TABLE_PREFIX = ""  # e.g., "RAW_"
LOAD_AS_IS_AS_STRING = True  # create all target columns as STRING to avoid type-cast drops
ENFORCE_TARGET_SCHEMA = True  # when True, use explicit target DDL for known tables

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"SQLite DB: {SQLITE_DB_PATH}")
if UPLOADED_DB_PATH is not None:
    print("Using DB uploaded via Colab.")
print(f"Export dir: {EXPORT_DIR.resolve()}")
print(f"Using Snowflake connection profile: {ACTIVE_CONNECTION}")


SQLite DB: /content/ISE558finaldb.db
Using DB uploaded via Colab.
Export dir: /content/sqlite_exports
Using Snowflake connection profile: my_example_connection


## 4) Helper Functions

### Utility functions for quoting, schema mapping, and DDL generation


In [5]:
def quote_ident(name: str) -> str:
    return '"' + name.replace('\"', '\\\"') + '"'


def sqlite_decltype_to_snowflake(decltype: str) -> str:
    if decltype is None:
        return "STRING"
    t = decltype.strip().upper()
    if "INT" in t:
        return "NUMBER"
    if any(x in t for x in ["REAL", "FLOA", "DOUB"]):
        return "FLOAT"
    if any(x in t for x in ["NUMERIC", "DECIMAL", "BOOLEAN"]):
        return "NUMBER"
    if any(x in t for x in ["DATE", "TIME"]):
        return "TIMESTAMP_NTZ"
    if "BLOB" in t:
        return "BINARY"
    return "STRING"


def discover_sqlite_tables(conn: sqlite3.Connection):
    q = """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """
    return [row[0] for row in conn.execute(q).fetchall()]


def get_table_columns(conn: sqlite3.Connection, table_name: str):
    # PRAGMA table_info columns: cid, name, type, notnull, dflt_value, pk
    rows = conn.execute(f"PRAGMA table_info({quote_ident(table_name)})").fetchall()
    return [{"name": r[1], "type": r[2]} for r in rows]


def create_snowflake_table_sql(target_table: str, columns: list[dict]) -> str:
    col_defs = []
    for col in columns:
        sf_type = sqlite_decltype_to_snowflake(col.get("type"))
        col_defs.append(f"{quote_ident(col['name'])} {sf_type}")
    cols_sql = ",\n    ".join(col_defs) if col_defs else '"_RAW" STRING'
    return f"CREATE OR REPLACE TABLE {quote_ident(target_table)} (\n    {cols_sql}\n)"

def create_snowflake_table_sql_all_string(target_table: str, columns: list[dict]) -> str:
    col_defs = []
    for col in columns:
        col_defs.append(f"{quote_ident(col['name'])} STRING")
    cols_sql = ",\n    ".join(col_defs) if col_defs else '"_RAW" STRING'
    return f"CREATE OR REPLACE TABLE {quote_ident(target_table)} (\n    {cols_sql}\n)"


def create_enforced_schema_sql(target_table: str) -> str | None:
    ddl = {
        "Client": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            clientID INTEGER AUTOINCREMENT PRIMARY KEY,
            client_name VARCHAR(255) NOT NULL,
            locationID INTEGER NOT NULL,
            phone_number VARCHAR(50),
            email VARCHAR(255) NOT NULL,
            last_update TIMESTAMP_NTZ
        )""",
        "Location": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            locationID INTEGER AUTOINCREMENT PRIMARY KEY,
            state VARCHAR(100),
            city VARCHAR(100)
        )""",
        "BusinessUnit": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            businessUnitID INTEGER AUTOINCREMENT PRIMARY KEY,
            business_unit_name VARCHAR(255) NOT NULL
        )""",
        "Project": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            projectID VARCHAR(10) PRIMARY KEY,
            created_at TIMESTAMP_NTZ,
            clientID INTEGER NOT NULL,
            unitID INTEGER NOT NULL,
            name VARCHAR(255) NOT NULL,
            type VARCHAR(50) NOT NULL,
            price NUMBER(10,2),
            estimated_budget NUMBER(10,2),
            planned_hours NUMBER(10,1),
            planned_start_date DATE NOT NULL,
            planned_end_date DATE NOT NULL,
            status VARCHAR(50) NOT NULL DEFAULT 'Not Started',
            actual_start_date DATE,
            actual_end_date DATE,
            progress NUMBER(10,1),
            last_update TIMESTAMP_NTZ
        )""",
        "Title": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            TitleID VARCHAR(10) PRIMARY KEY,
            Title VARCHAR(255) NOT NULL
        )""",
        "ProjectBillingRate": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            ProjectID VARCHAR(10) NOT NULL,
            TitleID VARCHAR(10) NOT NULL,
            Rate NUMBER(10,2) NOT NULL,
            PRIMARY KEY (projectID, TitleID)
        )""",
        "Deliverable": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            deliverableID VARCHAR(10) PRIMARY KEY,
            projectID VARCHAR(10) NOT NULL,
            name VARCHAR(255) NOT NULL,
            created_at TIMESTAMP_NTZ,
            price NUMBER(10,2),
            planned_start_date DATE NOT NULL,
            actual_start_date DATE,
            planned_hours NUMBER(10,1),
            due_date TIMESTAMP_NTZ NOT NULL,
            status VARCHAR(50) NOT NULL DEFAULT 'Pending',
            progress NUMBER(5,2) DEFAULT 0.00,
            submission_date TIMESTAMP_NTZ,
            invoiced_date TIMESTAMP_NTZ,
            last_update TIMESTAMP_NTZ
        )""",
        "Consultant": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            consultantID VARCHAR(10) PRIMARY KEY,
            businessUnitID INTEGER NOT NULL,
            first_name VARCHAR(255) NOT NULL,
            last_name VARCHAR(255),
            email VARCHAR(255) NOT NULL,
            contact VARCHAR(255),
            hire_year INTEGER,
            last_update TIMESTAMP_NTZ
        )""",
        "ConsultantDeliverable": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            recordID VARCHAR(10) PRIMARY KEY,
            consultantID VARCHAR(10) NOT NULL,
            deliverableID VARCHAR(10) NOT NULL,
            date TIMESTAMP_NTZ NOT NULL,
            hours NUMBER(10,2) NOT NULL,
            last_update TIMESTAMP_NTZ
        )""",
        "ConsultantTitleHistory": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            recordID VARCHAR(10) PRIMARY KEY,
            consultantID VARCHAR(10) NOT NULL,
            titleID VARCHAR(10) NOT NULL,
            start_date TIMESTAMP_NTZ NOT NULL,
            event_type VARCHAR(255),
            salary NUMBER(10,2) NOT NULL,
            last_update TIMESTAMP_NTZ
        )""",
        "Payroll": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            recordID VARCHAR(10) PRIMARY KEY,
            consultantID VARCHAR(10) NOT NULL,
            amount NUMBER(10,2),
            effective_date TIMESTAMP_NTZ
        )""",
        "ProjectTeam": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            projectID VARCHAR(10) NOT NULL,
            consultantID VARCHAR(10) NOT NULL,
            role VARCHAR(255),
            start_date TIMESTAMP_NTZ,
            end_date TIMESTAMP_NTZ,
            PRIMARY KEY (projectID, consultantID)
        )""",
        "ProjectExpense": f"""CREATE OR REPLACE TABLE {quote_ident(target_table)} (
            recordID VARCHAR(10) PRIMARY KEY,
            projectID VARCHAR(10) NOT NULL,
            deliverableID VARCHAR(10),
            date TIMESTAMP_NTZ NOT NULL,
            amount NUMBER(10,2) NOT NULL,
            description STRING NOT NULL,
            category VARCHAR(255),
            is_billable INTEGER
        )""",
    }
    return ddl.get(target_table)


## 5) Extract from SQLite

### Discover tables and export to CSV


In [6]:
if not SQLITE_DB_PATH.exists():
    raise FileNotFoundError(f"SQLite DB not found: {SQLITE_DB_PATH}")

sqlite_conn = sqlite3.connect(SQLITE_DB_PATH)
all_tables = discover_sqlite_tables(sqlite_conn)
tables = TABLES_TO_LOAD if TABLES_TO_LOAD else all_tables

if not tables:
    raise ValueError("No tables found in SQLite database.")

print(f"Discovered tables ({len(all_tables)}): {all_tables}")
print(f"Tables to load ({len(tables)}): {tables}")

exports = []
for table in tables:
    df = pd.read_sql_query(f"SELECT * FROM {quote_ident(table)}", sqlite_conn)
    csv_path = EXPORT_DIR / f"{table}.csv"
    df.to_csv(csv_path, index=False)
    schema_cols = get_table_columns(sqlite_conn, table)
    exports.append({
        "source_table": table,
        "target_table": f"{TARGET_TABLE_PREFIX}{table}",
        "csv_path": csv_path,
        "columns": schema_cols
    })
    print(f"Exported {table}: {len(df):,} rows -> {csv_path}")

sqlite_conn.close()

Discovered tables (13): ['BusinessUnit', 'Client', 'Consultant', 'ConsultantDeliverable', 'ConsultantTitleHistory', 'Deliverable', 'Location', 'Payroll', 'Project', 'ProjectBillingRate', 'ProjectExpense', 'ProjectTeam', 'Title']
Tables to load (13): ['BusinessUnit', 'Client', 'Consultant', 'ConsultantDeliverable', 'ConsultantTitleHistory', 'Deliverable', 'Location', 'Payroll', 'Project', 'ProjectBillingRate', 'ProjectExpense', 'ProjectTeam', 'Title']
Exported BusinessUnit: 4 rows -> sqlite_exports/BusinessUnit.csv
Exported Client: 355 rows -> sqlite_exports/Client.csv
Exported Consultant: 108 rows -> sqlite_exports/Consultant.csv
Exported ConsultantDeliverable: 6,507 rows -> sqlite_exports/ConsultantDeliverable.csv
Exported ConsultantTitleHistory: 112 rows -> sqlite_exports/ConsultantTitleHistory.csv
Exported Deliverable: 113 rows -> sqlite_exports/Deliverable.csv
Exported Location: 40 rows -> sqlite_exports/Location.csv
Exported Payroll: 638 rows -> sqlite_exports/Payroll.csv
Exported

## 5.1) Optional Additional Files

### Add Excel/JSON sources (GitHub + manual upload)
This step adds:
- `indirect_costs.xlsx` (downloaded from GitHub)
- `non_billable.xlsx` (uploaded manually in Colab)
- `client_feedback_initial.json` (downloaded from GitHub)



In [7]:
# Download GitHub files
for url in [INDIRECT_COSTS_XLSX_URL, CLIENT_FEEDBACK_JSON_URL]:
    file_name = url.split('/')[-1]
    out_path = EXTRA_SOURCE_DIR / file_name
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f"Downloaded: {out_path}")

# Optional manual upload for non_billable.xlsx in Colab
manual_non_billable_path = EXTRA_SOURCE_DIR / MANUAL_NON_BILLABLE_FILENAME
if not manual_non_billable_path.exists():
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        if MANUAL_NON_BILLABLE_FILENAME in uploaded:
            manual_non_billable_path.write_bytes(uploaded[MANUAL_NON_BILLABLE_FILENAME])
            print(f"Uploaded: {manual_non_billable_path}")
        elif uploaded:
            first_name = next(iter(uploaded.keys()))
            manual_non_billable_path.write_bytes(uploaded[first_name])
            print(f"Uploaded (renamed): {manual_non_billable_path}")
        else:
            print("No manual file uploaded; skipping non_billable.xlsx")
    except Exception:
        print("Not in Colab upload context; place non_billable.xlsx under ./extra_sources to include it.")

# Convert extra files to CSV and append to exports
indirect_xlsx_path = EXTRA_SOURCE_DIR / 'indirect_costs.xlsx'
if indirect_xlsx_path.exists():
    df_indirect = pd.read_excel(indirect_xlsx_path)
    out_csv = EXPORT_DIR / 'IndirectCosts.csv'
    df_indirect.to_csv(out_csv, index=False)
    exports.append({
        'source_table': 'indirect_costs.xlsx',
        'target_table': f'{TARGET_TABLE_PREFIX}IndirectCosts',
        'csv_path': out_csv,
        'columns': [{'name': c, 'type': 'TEXT'} for c in df_indirect.columns]
    })
    print(f"Prepared extra source -> {out_csv} ({len(df_indirect):,} rows)")

if manual_non_billable_path.exists():
    df_non_billable = pd.read_excel(manual_non_billable_path)
    out_csv = EXPORT_DIR / 'NonBillable.csv'
    df_non_billable.to_csv(out_csv, index=False)
    exports.append({
        'source_table': 'non_billable.xlsx',
        'target_table': f'{TARGET_TABLE_PREFIX}NonBillable',
        'csv_path': out_csv,
        'columns': [{'name': c, 'type': 'TEXT'} for c in df_non_billable.columns]
    })
    print(f"Prepared extra source -> {out_csv} ({len(df_non_billable):,} rows)")

feedback_json_path = EXTRA_SOURCE_DIR / 'client_feedbacks.json'
if feedback_json_path.exists():
    payload = json.loads(feedback_json_path.read_text(encoding='utf-8'))
    if isinstance(payload, dict):
        if 'data' in payload and isinstance(payload['data'], list):
            data = payload['data']
        else:
            data = [payload]
    elif isinstance(payload, list):
        data = payload
    else:
        data = [{'raw_value': payload}]
    df_feedback = pd.json_normalize(data)
    df_feedback = df_feedback.explode("responses", ignore_index=True)

    resp_cols = pd.json_normalize(df_feedback["responses"]).add_prefix("response_")
    df_feedback = pd.concat([df_feedback.drop(columns=["responses"]), resp_cols], axis=1)
    out_csv = EXPORT_DIR / 'ClientFeedbackInitial.csv'
    df_feedback.to_csv(out_csv, index=False)
    exports.append({
        'source_table': 'client_feedback_initial.json',
        'target_table': f'{TARGET_TABLE_PREFIX}ClientFeedbackInitial',
        'csv_path': out_csv,
        'columns': [{'name': c, 'type': 'TEXT'} for c in df_feedback.columns]
    })
    print(f"Prepared extra source -> {out_csv} ({len(df_feedback):,} rows)")

print(f"Total datasets queued for load: {len(exports)}")


Downloaded: extra_sources/indirect_costs.xlsx
Downloaded: extra_sources/client_feedbacks.json


Saving Non_Billable hours Fianalized one by Vikarm.xlsx to Non_Billable hours Fianalized one by Vikarm.xlsx
Uploaded (renamed): extra_sources/non_billable.xlsx
Prepared extra source -> sqlite_exports/IndirectCosts.csv (11 rows)
Prepared extra source -> sqlite_exports/NonBillable.csv (658 rows)
Prepared extra source -> sqlite_exports/ClientFeedbackInitial.csv (84 rows)
Total datasets queued for load: 16


## 6) Connect to Snowflake

### Validate config and create internal stage


### Troubleshooting SSO/SAML connection errors
- Ensure `account` is your Snowflake **account identifier** (not full URL).
- For browser-based SSO, set `authenticator = "externalbrowser"`.
- Keep `warehouse`, `database`, and `schema` empty if you want notebook auto-creation.


In [8]:
required = ["account", "user"]
missing = [
    k for k in required
    if not SNOWFLAKE_CONFIG.get(k) or str(SNOWFLAKE_CONFIG[k]).startswith("<")
]
if missing:
    raise ValueError(f"Missing Snowflake config values: {missing}")

if not SNOWFLAKE_CONFIG.get("password") and not SNOWFLAKE_CONFIG.get("authenticator"):
    raise ValueError("Provide either 'password' or 'authenticator' in SNOWFLAKE_CONFIG.")

valid_authenticators = {"externalbrowser", "snowflake", "oauth", "username_password_mfa", "SNOWFLAKE_JWT"}
auth_val = SNOWFLAKE_CONFIG.get("authenticator")
if auth_val and auth_val not in valid_authenticators:
    raise ValueError(
        f"Unsupported authenticator '{auth_val}'. Use one of: {sorted(valid_authenticators)}"
    )

sf_conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
sf_cur = sf_conn.cursor()

import time
suffix = str(int(time.time()))
warehouse = SNOWFLAKE_CONFIG.get("warehouse") or f"{RESOURCE_NAME_PREFIX}_WH_{suffix}"
database = SNOWFLAKE_CONFIG.get("database") or f"{RESOURCE_NAME_PREFIX}_DB_{suffix}"
schema = SNOWFLAKE_CONFIG.get("schema") or "PUBLIC"

if CREATE_RESOURCES_IF_MISSING:
    create_wh_sql = f"""
    CREATE WAREHOUSE IF NOT EXISTS {quote_ident(warehouse)}
    WAREHOUSE_SIZE = '{WAREHOUSE_SIZE}'
    WAREHOUSE_TYPE = '{WAREHOUSE_TYPE}'
    AUTO_SUSPEND = {WAREHOUSE_AUTO_SUSPEND}
    AUTO_RESUME = {'TRUE' if WAREHOUSE_AUTO_RESUME else 'FALSE'}
    INITIALLY_SUSPENDED = TRUE
    """
    sf_cur.execute(create_wh_sql)
    sf_cur.execute(f"CREATE DATABASE IF NOT EXISTS {quote_ident(database)}")
    sf_cur.execute(f"CREATE SCHEMA IF NOT EXISTS {quote_ident(database)}.{quote_ident(schema)}")
    print(f"Ensured warehouse/database/schema exist: {warehouse}, {database}, {schema}")

sf_cur.execute(f"USE WAREHOUSE {quote_ident(warehouse)}")
sf_cur.execute(f"USE DATABASE {quote_ident(database)}")
sf_cur.execute(f"USE SCHEMA {quote_ident(schema)}")
sf_cur.execute(f"CREATE OR REPLACE STAGE {quote_ident(STAGE_NAME)}")
print(f"Stage ready: {STAGE_NAME}")


Ensured warehouse/database/schema exist: SQLITE_INGEST_WH_1772163836, SQLITE_INGEST_DB_1772163836, PUBLIC
Stage ready: SQLITE_INGEST_STAGE


## 7) Load into Snowflake

### Create tables, PUT files, and COPY data


In [9]:
load_summary = []

for item in exports:
    source_table = item['source_table']
    target_table = item['target_table']
    csv_path = item['csv_path']
    columns = item['columns']

    enforced_sql = create_enforced_schema_sql(target_table) if ENFORCE_TARGET_SCHEMA else None
    create_sql = enforced_sql or (
        create_snowflake_table_sql_all_string(target_table, columns)
        if LOAD_AS_IS_AS_STRING else create_snowflake_table_sql(target_table, columns)
    )
    sf_cur.execute(create_sql)

    put_sql = f"PUT file://{csv_path.resolve()} @{quote_ident(STAGE_NAME)} AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
    sf_cur.execute(put_sql)

    copy_sql = f"""
    COPY INTO {quote_ident(target_table)}
    FROM @{quote_ident(STAGE_NAME)}/{csv_path.name}.gz
    FILE_FORMAT = (
        TYPE = CSV
        FIELD_OPTIONALLY_ENCLOSED_BY = '\"'
        SKIP_HEADER = 1
        EMPTY_FIELD_AS_NULL = TRUE
        NULL_IF = ('', 'NULL', 'None')
    )
    ON_ERROR = 'ABORT_STATEMENT'
    """
    sf_cur.execute(copy_sql)

    count_sql = f"SELECT COUNT(*) FROM {quote_ident(target_table)}"
    loaded_rows = sf_cur.execute(count_sql).fetchone()[0]

    load_summary.append({
        "source_table": source_table,
        "target_table": target_table,
        "rows_loaded": loaded_rows,
        "csv": str(csv_path),
    })

    print(f"Loaded {source_table} -> {target_table}: {loaded_rows:,} rows")

Loaded BusinessUnit -> BusinessUnit: 4 rows
Loaded Client -> Client: 355 rows
Loaded Consultant -> Consultant: 108 rows
Loaded ConsultantDeliverable -> ConsultantDeliverable: 6,507 rows
Loaded ConsultantTitleHistory -> ConsultantTitleHistory: 112 rows
Loaded Deliverable -> Deliverable: 113 rows
Loaded Location -> Location: 40 rows
Loaded Payroll -> Payroll: 638 rows
Loaded Project -> Project: 24 rows
Loaded ProjectBillingRate -> ProjectBillingRate: 42 rows
Loaded ProjectExpense -> ProjectExpense: 72 rows
Loaded ProjectTeam -> ProjectTeam: 173 rows
Loaded Title -> Title: 6 rows
Loaded indirect_costs.xlsx -> IndirectCosts: 11 rows
Loaded non_billable.xlsx -> NonBillable: 658 rows
Loaded client_feedback_initial.json -> ClientFeedbackInitial: 84 rows


## 8) Validate Results

### Review loaded row counts


In [10]:
pd.DataFrame(load_summary).sort_values("target_table")

,source_table,target_table,rows_loaded,csv
0,BusinessUnit,BusinessUnit,4,sqlite_exports/BusinessUnit.csv
1,Client,Client,355,sqlite_exports/Client.csv
15,client_feedback_initial.json,ClientFeedbackInitial,84,sqlite_exports/ClientFeedbackInitial.csv
2,Consultant,Consultant,108,sqlite_exports/Consultant.csv
3,ConsultantDeliverable,ConsultantDeliverable,6507,sqlite_exports/ConsultantDeliverable.csv
4,ConsultantTitleHistory,ConsultantTitleHistory,112,sqlite_exports/ConsultantTitleHistory.csv
5,Deliverable,Deliverable,113,sqlite_exports/Deliverable.csv
13,indirect_costs.xlsx,IndirectCosts,11,sqlite_exports/IndirectCosts.csv
6,Location,Location,40,sqlite_exports/Location.csv
14,non_billable.xlsx,NonBillable,658,sqlite_exports/NonBillable.csv


## 9) Cleanup

### Close Snowflake connection


In [11]:
sf_cur.close()
sf_conn.close()
print("Snowflake connection closed.")

Snowflake connection closed.
